In [50]:
# Original BERT code:
from transformers import BertTokenizer, BertModel

# RoBERTa code (commented out):
# from transformers import RobertaTokenizer, RobertaModel

# GPT-2 code (commented out):
# from transformers import GPT2Tokenizer, GPT2Model
from datasets import load_from_disk
from evaluate import load
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"
#  You can install and import any other libraries if needed

In [51]:
# Some Chinese punctuations will be tokenized as [UNK], so we replace them with English ones
token_replacement = [
    ["：" , ":"],
    ["，" , ","],
    ["“" , "\""],
    ["”" , "\""],
    ["？" , "?"],
    ["……" , "..."],
    ["！" , "!"]
]

In [52]:

# Original BERT code:
tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased", cache_dir="./cache/")

# RoBERTa code (commented out):
# tokenizer = RobertaTokenizer.from_pretrained("roberta-base", cache_dir="./cache/")

# GPT-2 code (commented out):
# tokenizer = GPT2Tokenizer.from_pretrained("gpt2", cache_dir="./cache/")
# tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have a pad token, use eos_token

In [53]:
class SemevalDataset(Dataset):
    def __init__(self, split="train") -> None:
        super().__init__()
        assert split in ["train", "validation", "test"]
        # Load from local dataset directory
        dataset_dict = load_from_disk("./data/sem_eval_2014_task_1")
        self.data = dataset_dict[split].to_list()

    def __getitem__(self, index):
        d = self.data[index]
        # Replace Chinese punctuations with English ones
        for k in ["premise", "hypothesis"]:
            for tok in token_replacement:	
                d[k] = d[k].replace(tok[0], tok[1])
        return d

    def __len__(self):
        return len(self.data)

data_sample = SemevalDataset(split="train").data[:3]
print(f"Dataset example: \n{data_sample[0]} \n{data_sample[1]} \n{data_sample[2]}")

Dataset example: 
{'sentence_pair_id': 1, 'premise': 'A group of kids is playing in a yard and an old man is standing in the background', 'hypothesis': 'A group of boys in a yard is playing and a man is standing in the background', 'relatedness_score': 4.5, 'entailment_judgment': 0} 
{'sentence_pair_id': 2, 'premise': 'A group of children is playing in the house and there is no man standing in the background', 'hypothesis': 'A group of kids is playing in a yard and an old man is standing in the background', 'relatedness_score': 3.200000047683716, 'entailment_judgment': 0} 
{'sentence_pair_id': 3, 'premise': 'The young boys are playing outdoors and the man is smiling nearby', 'hypothesis': 'The kids are playing outdoors near a man with a smile', 'relatedness_score': 4.699999809265137, 'entailment_judgment': 1}


In [54]:
# Define the hyperparameters
# You can modify these values if needed
lr = 5e-6
epochs = 20
train_batch_size = 8
validation_batch_size = 8

In [55]:
# TODO1: Create batched data for DataLoader
# `collate_fn` is a function that defines how the data batch should be packed.
# This function will be called in the DataLoader to pack the data batch.

def collate_fn(batch):
    # TODO1-1: Implement the collate_fn function
    # Write your code here
    # The input parameter is a data batch (tuple), and this function packs it into tensors.
    # Use tokenizer to pack tokenize and pack the data and its corresponding labels.
    # Return the data batch and labels for each sub-task.
    premises = [d['premise'] for d in batch]
    hypotheses = [d['hypothesis'] for d in batch]
    inputs = tokenizer(premises, hypotheses, padding=True, truncation=True, return_tensors="pt")

    entailment_labels = [d['entailment_judgment'] for d in batch]
    relatedness_scores = [d['relatedness_score'] for d in batch]

    entailment_labels_ids = torch.tensor(entailment_labels, dtype=torch.long)
    relatedness_scores_tensor = torch.tensor(relatedness_scores, dtype=torch.float)
    
    batch = {}
    batch["input_ids"] = inputs.input_ids
    batch["attention_mask"] = inputs.attention_mask
    batch["entailment"] = entailment_labels_ids
    batch["relatedness"] = relatedness_scores_tensor
    return batch

# TODO1-2: Define your DataLoader
dl_train = DataLoader(SemevalDataset("train"), batch_size=train_batch_size, collate_fn=collate_fn)
dl_validation = DataLoader(SemevalDataset("validation"), batch_size=validation_batch_size, collate_fn=collate_fn)
dl_test = DataLoader(SemevalDataset("test"), batch_size=validation_batch_size, collate_fn=collate_fn)

In [ ]:
# TODO2: Construct your model
class MultiLabelModel(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Write your code here
        # Define what modules you will use in the model
        # Please use "google-bert/bert-base-uncased" model (https://huggingface.co/google-bert/bert-base-uncased)
        # Besides the base model, you may design additional architectures by incorporating linear layers, activation functions, or other neural components.
        # Remark: The use of any additional pretrained language models is not permitted.
        self.bert = BertModel.from_pretrained("google-bert/bert-base-uncased", cache_dir="./cache/")
        # self.roberta = RobertaModel.from_pretrained("roberta-base", cache_dir="./cache/")
        # self.gpt2 = GPT2Model.from_pretrained("gpt2", cache_dir="./cache/")
        self.dropout = torch.nn.Dropout(0.1)
        
        self.entailment_classifier = torch.nn.Linear(768, 3)
        
        self.relatedness_regressor = torch.nn.Linear(768, 1)
        
    def forward(self, **kwargs):
        # Write your code here
        # Forward pass
        input_ids = kwargs.get("input_ids")
        attention_mask = kwargs.get("attention_mask")
        
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        # outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)
        
        pooled_output = outputs.pooler_output  
        # pooled_output = outputs.last_hidden_state[:, 0, :]  
        # GPT-2: use mean pooling over sequence length (masked)
        # mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
        # sum_hidden = torch.sum(outputs.last_hidden_state * mask_expanded, dim=1)
        # sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        # pooled_output = sum_hidden / sum_mask 
        pooled_output = self.dropout(pooled_output)
        
        entailment_logits = self.entailment_classifier(pooled_output)  # Shape: (batch_size, 3)
        relatedness_score = self.relatedness_regressor(pooled_output).squeeze(-1)  # Shape: (batch_size,)
        
        return {
            "entailment_logits": entailment_logits,
            "relatedness_score": relatedness_score
        }

In [57]:
# TODO3: Define your optimizer and loss function

model = MultiLabelModel().to(device)
# TODO3-1: Define your Optimizer
optimizer = AdamW(model.parameters(), lr=lr)

# TODO3-2: Define your loss functions (you should have two)
# Write your code here
criterion_entailment = torch.nn.CrossEntropyLoss()
criterion_relatedness = torch.nn.MSELoss()
# scoring functions
psr = load("pearsonr")
acc = load("accuracy")

In [58]:
best_score = 0.0
for ep in range(epochs):
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train()
    # TODO4: Write the training loop
    # Write your code here
    # train your model
    # clear gradient
    # forward pass
    # compute loss
    # back-propagation
    # model optimization
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        
        loss_ent = criterion_entailment(outputs["entailment_logits"], batch["entailment"])
        loss_rel = criterion_relatedness(outputs["relatedness_score"], batch["relatedness"])
        
        total_loss = loss_ent + loss_rel
        
        total_loss.backward()
        optimizer.step()
        
        pbar.set_postfix({"loss_ent": loss_ent.item(), "loss_rel": loss_rel.item()})

    pbar = tqdm(dl_validation)
    pbar.set_description(f"Validation epoch [{ep+1}/{epochs}]")
    model.eval()
    # TODO5: Write the evaluation loop
    
    # Lists to collect all predictions and labels
    all_entailment_preds = []
    all_entailment_labels = []
    all_relatedness_preds = []
    all_relatedness_labels = []
    
    with torch.no_grad(): 
        for batch in pbar:

            batch = {k: v.to(device) for k, v in batch.items()}
            
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            
            entailment_preds = torch.argmax(outputs["entailment_logits"], dim=1)
            
            relatedness_preds = outputs["relatedness_score"]
            
            all_entailment_preds.extend(entailment_preds.cpu().tolist())
            all_entailment_labels.extend(batch["entailment"].cpu().tolist())
            all_relatedness_preds.extend(relatedness_preds.cpu().tolist())
            all_relatedness_labels.extend(batch["relatedness"].cpu().tolist())
    
    # Compute evaluation metrics
    pearson_corr = psr.compute(predictions=all_relatedness_preds, references=all_relatedness_labels)['pearsonr']
    accuracy = acc.compute(predictions=all_entailment_preds, references=all_entailment_labels)['accuracy']
    
    # Print results
    print(f"Epoch {ep+1}/{epochs} - Pearson Correlation: {pearson_corr:.4f}, Accuracy: {accuracy:.4f}")
    
    if pearson_corr + accuracy > best_score:
        best_score = pearson_corr + accuracy
        torch.save(model.state_dict(), f'./saved_models/best_model_bert.ckpt')

Validation epoch [1/20]: 100%|██████████| 63/63 [00:01<00:00, 61.81it/s]


Epoch 1/20 - Pearson Correlation: 0.6766, Accuracy: 0.5660


Validation epoch [2/20]: 100%|██████████| 63/63 [00:01<00:00, 61.18it/s]


Epoch 2/20 - Pearson Correlation: 0.7658, Accuracy: 0.6840


Validation epoch [3/20]: 100%|██████████| 63/63 [00:01<00:00, 61.17it/s]


...

Epoch 18/20 - Pearson Correlation: 0.8574, Accuracy: 0.8380


Validation epoch [19/20]: 100%|██████████| 63/63 [00:01<00:00, 60.97it/s]


Epoch 19/20 - Pearson Correlation: 0.8617, Accuracy: 0.8380


Validation epoch [20/20]: 100%|██████████| 63/63 [00:01<00:00, 60.99it/s]


Epoch 20/20 - Pearson Correlation: 0.8573, Accuracy: 0.8520


In [ ]:
# Load the model
model = MultiLabelModel().to(device)
model.load_state_dict(torch.load(f"./saved_models/best_model_bert.ckpt", weights_only=True))

# Test Loop
pbar = tqdm(dl_test, desc="Test")
model.eval()

# TODO6: Write the test loop

all_entailment_preds = []
all_entailment_labels = []
all_relatedness_preds = []
all_relatedness_labels = []

with torch.no_grad():  
    for batch in pbar:
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        
        entailment_preds = torch.argmax(outputs["entailment_logits"], dim=1)
        
        relatedness_preds = outputs["relatedness_score"]
        
        all_entailment_preds.extend(entailment_preds.cpu().tolist())
        all_entailment_labels.extend(batch["entailment"].cpu().tolist())
        all_relatedness_preds.extend(relatedness_preds.cpu().tolist())
        all_relatedness_labels.extend(batch["relatedness"].cpu().tolist())

test_pearson_corr = psr.compute(predictions=all_relatedness_preds, references=all_relatedness_labels)['pearsonr']
test_accuracy = acc.compute(predictions=all_entailment_preds, references=all_entailment_labels)['accuracy']


print(f"{'='*50}")
print(f"Test Pearson Correlation: {test_pearson_corr:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"{'='*50}")

Test: 100%|██████████| 616/616 [00:09<00:00, 67.41it/s]

Test Pearson Correlation: 0.8661
Test Accuracy: 0.8616


In [ ]:
import pandas as pd

print("="*70)
print("ERROR ANALYSIS")
print("="*70)

# Reload test data
test_dataset = SemevalDataset("test")

errors = []
label_names = {0: 'Neutral', 1: 'Entailment', 2: 'Contradiction'}

model.eval()
with torch.no_grad():
    for i, sample in enumerate(test_dataset):
        premise = sample['premise']
        hypothesis = sample['hypothesis']
        inputs = tokenizer([premise], [hypothesis], padding=True, truncation=True, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
    
        outputs = model(**inputs)
        entailment_pred = torch.argmax(outputs["entailment_logits"], dim=1).item()
        relatedness_pred = outputs["relatedness_score"].item()
        
        entailment_true = sample['entailment_judgment']
        relatedness_true = sample['relatedness_score']
        
        if entailment_pred != entailment_true or abs(relatedness_pred - relatedness_true) > 1.5:
            errors.append({
                'premise': premise,
                'hypothesis': hypothesis,
                'premise_len': len(premise.split()),
                'hypothesis_len': len(hypothesis.split()),
                'entailment_true': label_names[entailment_true],
                'entailment_pred': label_names[entailment_pred],
                'entailment_error': entailment_pred != entailment_true,
                'relatedness_true': relatedness_true,
                'relatedness_pred': relatedness_pred,
                'relatedness_error': abs(relatedness_pred - relatedness_true)
            })

df_errors = pd.DataFrame(errors)

print(f"\n1. TOTAL ERRORS FOUND: {len(df_errors)}/{len(test_dataset)} samples")


print("\n2. SAMPLE ERRORS:")
print("-" * 70)
for i, row in df_errors.head(10).iterrows():
    print(f"\nExample {i+1}:")
    print(f"Premise: {row['premise'][:80]}...")
    print(f"Hypothesis: {row['hypothesis'][:80]}...")
    if row['entailment_error']:
        print(f"Entailment: {row['entailment_true']} → {row['entailment_pred']} (WRONG)")
    print(f"Relatedness: {row['relatedness_true']:.2f} → {row['relatedness_pred']:.2f} (error: {row['relatedness_error']:.2f})")
    print(f"Text lengths: Premise={row['premise_len']}, Hypothesis={row['hypothesis_len']}")

# Error patterns
print("\n" + "="*70)
print("3. ERROR PATTERNS IDENTIFIED:")
print("="*70)

# Pattern 1: Text length
avg_premise_len = df_errors['premise_len'].mean()
print(f"\nPattern 1 - Text Length:")
print(f"  - Average premise length in errors: {avg_premise_len:.1f} words")
print(f"  - Observation: {'Longer' if avg_premise_len > 15 else 'Shorter'} texts appear to be more challenging")

# Pattern 2: Confusion between labels
if df_errors['entailment_error'].sum() > 0:
    entailment_errors = df_errors[df_errors['entailment_error']]
    confusion = entailment_errors.groupby(['entailment_true', 'entailment_pred']).size().head(3)
    print(f"\nPattern 2 - Label Confusion:")
    for (true_label, pred_label), count in confusion.items():
        print(f"  - Often confuses '{true_label}' with '{pred_label}': {count} times")

# Pattern 3: Relatedness score ranges
print(f"\nPattern 3 - Relatedness Score Issues:")
score_ranges = {'Low (0-2)': (0, 2), 'Medium (2-4)': (2, 4), 'High (4-5)': (4, 5)}
for name, (low, high) in score_ranges.items():
    in_range = df_errors[(df_errors['relatedness_true'] >= low) & (df_errors['relatedness_true'] < high)]
    if len(in_range) > 0:
        avg_error = in_range['relatedness_error'].mean()
        print(f"  - {name} scores: {len(in_range)} errors, avg error = {avg_error:.2f}")



ERROR ANALYSIS

1. TOTAL ERRORS FOUND: 736/4927 samples

2. SAMPLE ERRORS:
----------------------------------------------------------------------

Example 1:
Premise: A brown dog is attacking another animal in front of the man in pants...
Hypothesis: A brown dog is helping another animal in front of the man in pants...
Entailment: Neutral → Entailment (WRONG)
Relatedness: 3.66 → 4.49 (error: 0.83)
Text lengths: Premise=14, Hypothesis=14

Example 2:
Premise: A person in a black jacket is doing tricks on a motorbike...
Hypothesis: A person on a black motorbike is doing tricks with a jacket...
Entailment: Neutral → Entailment (WRONG)
Relatedness: 3.00 → 4.75 (error: 1.75)
Text lengths: Premise=12, Hypothesis=12

Example 3:
Premise: The player is missing the basket and a crowd is in background...
Hypothesis: The player is dunking the basketball into the net and a crowd is in background...
Entailment: Contradiction → Neutral (WRONG)
Relatedness: 3.90 → 3.66 (error: 0.24)
Text lengths: Premi